# Gaussian figures for Lecture 1

Author-facing figure source for the univariate Gaussian and Book 2 §2.3 slides.
Run every cell from top to bottom. The SVG files are saved under
`generated/`. The original figures support `slide_deck.typ`; the three-operation
panels and geometry recap support `slide_deck_revised.typ`.

The plots are original course renderings of ideas in Murphy, *Probabilistic
Machine Learning: An Introduction* §2.6.2 and *Advanced Topics* §2.3.1.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("generated")
OUT.mkdir(exist_ok=True)
ACCENT = "#aa624d"
INK = "#302d2a"
MUTED = "#7a7771"
PAPER = "#fbfaf7"
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## 1. Univariate Gaussian spread

Keep the mean fixed at zero and vary the standard deviation. Each curve
integrates to one, so a narrower curve has a taller peak.

In [ ]:
x = np.linspace(-5, 5, 800)
fig, ax = plt.subplots(figsize=(8.8, 3.4), constrained_layout=True)
for sd, color in [(0.5, ACCENT), (1.0, INK), (2.0, MUTED)]:
    pdf = np.exp(-0.5 * (x / sd) ** 2) / (np.sqrt(2 * np.pi) * sd)
    ax.plot(x, pdf, lw=2.7, color=color, label=fr"$\sigma={sd:g}$")
ax.axvline(0, lw=1, color="#b6b1aa", ls="--")
ax.set(xlim=(-4.5, 4.5), ylim=(0, 0.9), xlabel="$y$", ylabel="Density")
ax.legend(frameon=False, ncol=3, loc="upper right")
ax.grid(alpha=0.15)
fig.savefig(OUT / "gaussian_spread.svg", bbox_inches="tight")
plt.close(fig)


## 2. Covariance and contour geometry

Equal Mahalanobis distance gives equal-density contours. Compare a full,
diagonal, and spherical covariance on the same coordinate scale.

In [ ]:
grid = np.linspace(-3.3, 3.3, 350)
xx, yy = np.meshgrid(grid, grid)
points = np.stack([xx, yy], axis=-1)
cases = [
    ("Full covariance", np.array([[1.4, 0.85], [0.85, 1.0]])),
    ("Diagonal covariance", np.diag([1.4, 0.55])),
    ("Spherical covariance", 0.85 * np.eye(2)),
]
fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.8), constrained_layout=True)
for ax, (title, cov) in zip(axes, cases):
    q = np.einsum("...i,ij,...j->...", points, np.linalg.inv(cov), points)
    ax.contour(xx, yy, q, levels=[1, 2.5, 5], colors=[ACCENT], linewidths=[2.2, 1.8, 1.4])
    ax.plot(0, 0, "o", color=INK, ms=4)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", title=title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_xticks([-3, 0, 3])
    ax.set_yticks([-3, 0, 3])
    ax.grid(alpha=0.12)
fig.savefig(OUT / "gaussian_contours.svg", bbox_inches="tight")
plt.close(fig)


## 3. High-dimensional Gaussian radius

For a standard Gaussian in $D$ dimensions, plot the simulated radius divided
by $\sqrt{D}$. Its relative spread narrows as dimension increases. Fixed
seed makes the plot reproducible.

In [ ]:
rng = np.random.default_rng(20261001)
fig, ax = plt.subplots(figsize=(8.8, 3.4), constrained_layout=True)
for d, color in [(2, MUTED), (20, INK), (100, ACCENT)]:
    samples = rng.standard_normal((60_000, d))
    radius_ratio = np.linalg.norm(samples, axis=1) / np.sqrt(d)
    ax.hist(radius_ratio, bins=95, range=(0, 2.3), density=True,
            histtype="step", lw=2.4, color=color, label=fr"$D={d}$")
ax.axvline(1, color="#b6b1aa", lw=1, ls="--")
ax.set(xlim=(0, 2.3), xlabel=r"Radius $R/\sqrt{D}$", ylabel="Density")
ax.legend(frameon=False, ncol=3, loc="upper right")
ax.grid(alpha=0.15)
fig.savefig(OUT / "gaussian_shell_radius.svg", bbox_inches="tight")
plt.close(fig)


## 4. Three operations on one joint Gaussian

These panels introduce the three operations before deriving their Gaussian rules.
Use the same bivariate normal with zero mean, unit marginal variances, and
correlation $\rho=0.8$ in all three panels. The contour levels and coordinate
scales match across panels.

- **Summarize:** mark the mean and show that covariance describes the cloud's shape.
- **Marginalize:** integrate a vertical column over every $x_2$ to obtain $p(x_1)$.
- **Condition:** fix $x_2=a=1.5$ and normalize that horizontal slice. The resulting
  density has mean $\rho a=1.2$ and variance $1-\rho^2=0.36$. The dashed curve is
  the marginal, retained for comparison.

The slide supplies the integrals as editable Typst math below the SVG panels.
Numerical integration checks the plotted marginal, conditional, and moments;
the checks concern this illustration, not an approximation needed for Gaussian inference.

In [ ]:
# One probability model for all three operations.
rho = 0.8
observed_x2 = 1.5
operation_mean = np.zeros(2)
operation_cov = np.array([[1.0, rho], [rho, 1.0]])
conditional_mean = rho * observed_x2
conditional_variance = 1.0 - rho**2

def operation_joint(x1, x2):
    q = (x1**2 - 2 * rho * x1 * x2 + x2**2) / (1 - rho**2)
    return np.exp(-q / 2) / (2 * np.pi * np.sqrt(1 - rho**2))

def normal_density(x, mean=0.0, variance=1.0):
    return np.exp(-(x - mean)**2 / (2 * variance)) / np.sqrt(2 * np.pi * variance)

operation_grid = np.linspace(-3.2, 3.2, 401)
op_x1, op_x2 = np.meshgrid(operation_grid, operation_grid)
op_q = (op_x1**2 - 2 * rho * op_x1 * op_x2 + op_x2**2) / (1 - rho**2)
operation_files = []

# Keep SVG identifiers deterministic and labels legible at slide-panel size.
with plt.rc_context({
    'font.size': 16, 'axes.labelsize': 18, 'xtick.labelsize': 13,
    'ytick.labelsize': 13, 'svg.hashsalt': 'l01-gaussian-operations',
    'text.color': INK, 'axes.labelcolor': INK,
}):
    for operation in ['summarize', 'marginalize', 'condition']:
        fig = plt.figure(figsize=(4.2, 3.2))
        # Equal physical size, limits, and contour levels for each joint cloud.
        bottom = 0.29 if operation == 'summarize' else 0.46
        ax = fig.add_axes([0.16, bottom, 0.77, 0.48])
        ax.contour(op_x1, op_x2, op_q, levels=[1, 2.5, 5],
                   colors=[ACCENT], linewidths=[1.9, 1.5, 1.1])
        ax.set(xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), ylabel='$x_2$')
        ax.set_xticks([-2, 0, 2])
        ax.set_yticks([-2, 0, 2])
        ax.tick_params(length=3, colors=MUTED)
        ax.spines[['left', 'bottom']].set_color(MUTED)

        if operation == 'summarize':
            ax.set_xlabel('$x_1$', labelpad=0)
            ax.plot(0, 0, 'o', color=INK, ms=6, zorder=4)
            ax.annotate(r'$\mu$', xy=(0, 0), xytext=(-2.15, 1.25),
                        fontsize=21, color=INK,
                        arrowprops={'arrowstyle': '->', 'color': INK, 'lw': 1.2})
            ax.annotate(r'$\Sigma$', xy=(1.4, 1.4), xytext=(2.05, -1.8),
                        fontsize=21, color=INK,
                        arrowprops={'arrowstyle': '->', 'color': INK, 'lw': 1.2})
            fig.text(0.55, 0.055, 'Center and spread', ha='center', color=INK)
        else:
            ax.tick_params(labelbottom=False)
            density_ax = fig.add_axes([0.16, 0.14, 0.77, 0.21])
            density_ax.set(xlim=(-3.2, 3.2), ylim=(0, 0.78))
            density_ax.set_xlabel('$x_1$', labelpad=0)
            density_ax.set_xticks([-2, 0, 2])
            density_ax.set_yticks([])
            density_ax.spines['left'].set_visible(False)
            density_ax.spines['bottom'].set_color(MUTED)
            density_ax.tick_params(length=3, colors=MUTED)
            marginal = normal_density(operation_grid)
            if operation == 'marginalize':
                # The shaded column illustrates integration, not a finite bin rule.
                ax.axvspan(0.35, 0.65, color=ACCENT, alpha=0.17)
                ax.annotate('', xy=(0.5, 3.05), xytext=(0.5, -3.05),
                            arrowprops={'arrowstyle': '<->', 'color': INK, 'lw': 1.2})
                density_ax.plot(operation_grid, marginal, color=ACCENT, lw=2.3)
                density_ax.fill_between(operation_grid, marginal, color=ACCENT, alpha=0.12)
                density_ax.plot(0.5, normal_density(0.5), 'o', color=INK, ms=5)
                density_ax.text(-3.05, 0.57, r'$p(x_1)$', color=ACCENT)
            else:
                ax.axhline(observed_x2, color=INK, ls='--', lw=1.2)
                ax.text(-3.0, observed_x2 + 0.23, '$x_2=a$', fontsize=14, color=INK,
                        bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 0.4})
                ax.plot(conditional_mean, observed_x2, 'o', color=INK, ms=5)
                conditional = normal_density(operation_grid, conditional_mean, conditional_variance)
                density_ax.plot(operation_grid, marginal, color=MUTED, ls='--', lw=1.4)
                density_ax.plot(operation_grid, conditional, color=ACCENT, lw=2.3)
                density_ax.fill_between(operation_grid, conditional, color=ACCENT, alpha=0.12)
                density_ax.text(-3.05, 0.57, r'$p(x_1\mid a)$', color=ACCENT)
                density_ax.text(-3.05, 0.23, '$p(x_1)$', fontsize=13, color=MUTED)

        filename = f'gaussian_operations__{operation}.svg'
        # Fixed canvas preserves panel geometry; date-free metadata makes reruns stable.
        fig.savefig(OUT / filename, transparent=True, metadata={'Date': None})
        plt.close(fig)
        operation_files.append(filename)


In [ ]:
# Integrate over a wide domain to verify the actual curves and summary moments.
integration_grid = np.linspace(-8, 8, 1601)
g1, g2 = np.meshgrid(integration_grid, integration_grid)
joint_values = operation_joint(g1, g2)
integrate_x2 = lambda values: np.trapezoid(values, integration_grid, axis=0)
integrate_joint = lambda values: np.trapezoid(integrate_x2(values), integration_grid)
np.testing.assert_allclose(integrate_joint(joint_values), 1, atol=1e-10)
np.testing.assert_allclose(integrate_x2(joint_values), normal_density(integration_grid), atol=1e-10)
numeric_mean = np.array([integrate_joint(g1 * joint_values), integrate_joint(g2 * joint_values)])
numeric_cov = np.array([
    [integrate_joint(g1**2 * joint_values), integrate_joint(g1 * g2 * joint_values)],
    [integrate_joint(g1 * g2 * joint_values), integrate_joint(g2**2 * joint_values)],
])
np.testing.assert_allclose(numeric_mean, operation_mean, atol=1e-10)
np.testing.assert_allclose(numeric_cov, operation_cov, atol=1e-10)
slice_values = operation_joint(integration_grid, observed_x2)
slice_mass = np.trapezoid(slice_values, integration_grid)
normalized_slice = slice_values / slice_mass
np.testing.assert_allclose(slice_mass, normal_density(observed_x2), atol=1e-10)
np.testing.assert_allclose(normalized_slice,
                           normal_density(integration_grid, conditional_mean, conditional_variance),
                           atol=1e-10)
print('Verified mean, covariance, marginal, and normalized conditional slice.')


## 5. Geometry recap: one cloud, several descriptions

Reuse the zero-mean joint Gaussian and covariance from the operation panels.
Mark its center, a point $\mathbf{x}=(1.5,1.5)$, and the displacement between them.
The point lies on the $d_\Sigma^2=2.5$ equal-density contour. The covariance
annotation points to the outer contour to connect the matrix with shape.
The slide supplies the correlation, distance, and affine formulas as editable
Typst math. This figure closes the geometry section before blockwise inference.


In [ ]:
# Reuse the same model and contour levels as the three-operation overview.
summary_point = np.array([1.5, 1.5])
summary_displacement = summary_point - operation_mean
summary_distance = summary_displacement @ np.linalg.solve(operation_cov, summary_displacement)
np.testing.assert_allclose(summary_distance, 2.5)
# The covariance annotation also terminates on the specified outer contour.
shape_point = np.array([1.0, -0.4])
np.testing.assert_allclose(shape_point @ np.linalg.solve(operation_cov, shape_point), 5.0)

with plt.rc_context({
    'font.size': 17, 'axes.labelsize': 19, 'xtick.labelsize': 13,
    'ytick.labelsize': 13, 'svg.hashsalt': 'l01-gaussian-geometry-summary',
    'text.color': INK, 'axes.labelcolor': INK,
}):
    fig, ax = plt.subplots(figsize=(5.3, 3.7))
    fig.subplots_adjust(left=0.12, right=0.98, bottom=0.16, top=0.97)
    ax.contour(op_x1, op_x2, op_q, levels=[1, 2.5, 5],
               colors=[ACCENT], linewidths=[1.5, 2.0, 1.5])
    ax.set(xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), xlabel='$x_1$', ylabel='$x_2$')
    ax.set_aspect('equal')
    ax.set_xticks([-2, 0, 2])
    ax.set_yticks([-2, 0, 2])
    ax.tick_params(length=3, colors=MUTED)
    ax.spines[['left', 'bottom']].set_color(MUTED)
    ax.plot(*operation_mean, 'o', color=INK, ms=6, zorder=4)
    ax.plot(*summary_point, 'o', color=INK, ms=5, zorder=4)
    ax.annotate('', xy=summary_point, xytext=operation_mean,
                arrowprops={'arrowstyle': '->', 'color': INK, 'lw': 1.4})
    ax.annotate(r'$\mu$', xy=operation_mean, xytext=(-2.4, 0.7), fontsize=20,
                arrowprops={'arrowstyle': '->', 'color': INK, 'lw': 1.1})
    ax.text(1.64, 1.53, r'$\mathbf{x}$', fontsize=20, color=INK)
    ax.annotate(r'$\Sigma$', xy=shape_point, xytext=(2.25, -1.55), fontsize=20,
                arrowprops={'arrowstyle': '->', 'color': INK, 'lw': 1.1})
    ax.annotate(r'$d_\Sigma^2=2.5$', xy=-summary_point, xytext=(-3.1, -2.9),
                fontsize=16, color=ACCENT,
                arrowprops={'arrowstyle': '->', 'color': ACCENT, 'lw': 1.1})
    summary_filename = 'gaussian_geometry__summary.svg'
    fig.savefig(OUT / summary_filename, transparent=True, metadata={'Date': None})
    plt.close(fig)
print(f'Geometry recap: marked point has squared Mahalanobis distance {summary_distance:.1f}.')


## Checks

The seven generated files should exist and be nonempty. The radius figure
shows concentration **relative to** $\sqrt{D}$; it does not claim the
unnormalized radial width vanishes.

In [ ]:
for name in ["gaussian_spread.svg", "gaussian_contours.svg", "gaussian_shell_radius.svg", *operation_files, summary_filename]:
    file = OUT / name
    assert file.exists() and file.stat().st_size > 1000, file
    print(file, file.stat().st_size, "bytes")
